# Data Crawling pipeline

We need to crawl **labelled data** (w subtitle) and **unlabelled data** (w/o subtitle).

### 1) Download video
- https://github.com/yt-dlp/yt-dlp
- Start with a text file (to_download.txt) containing videos/playlists to download (e.g., https://www.youtube.com/watch?v=bRovJsvW-lk)
- Download videos to this folder (downloaded)

**Possible sources:** YouTube, bilibili

**Some playlists with Chinese subtitles below found:**
* https://www.youtube.com/watch?v=bRovJsvW-lk&list=PLsi-maNi24hH6hK7bt9btvfig1HzSHXEe
* https://www.youtube.com/watch?v=A-QJsVWmDBQ&list=PLD7r7QkgdBxBc2Rdy25P9WLIMXHDGNkf_&index=1
* https://www.youtube.com/watch?v=K4grKMQY4jY&list=PLD7r7QkgdBxCC67uetBnDV_vA6vPDA_qx&index=2

**Youtube channels (can find playlists here):**
* https://www.youtube.com/@%E6%BD%AE%E6%B1%95%E6%96%87%E5%8C%96/playlists
* https://www.youtube.com/@1.allantan785/videos

**Another possible source:** https://www.mewatch.sg/list/Chinese-Dialect-Others-291831?order=latest-release&audio_language=teochew (BUT this is teochew opera...)

### 2) Extract Audio Segments
- https://github.com/Zulko/moviepy
- https://github.com/snakers4/silero-vad
- **are there better ways? better models? better preprocessing?**

### 3) Extract Hard-coded Subtitles
- https://github.com/PaddlePaddle/PaddleOCR
- **are there better ways? better models? better preprocessing?**

### 4) Handling Data
- Need to handle files (currently have video, audio, and cropped frames). **What better way to store the data?**
- At the end of the day, we only need **the original audio, the timestamps, extracted subtitles (if available).**

In [ ]:
!mkdir downloaded

In [ ]:
!rm -rf downloaded/*

In [ ]:
!pip install -U yt-dlp
!pip install -U moviepy
!pip install -U silero-vad
# cpu
!pip install paddlepaddle==3.0.0b1
# gpu
# !pip install paddlepaddle-gpu
!pip install paddleocr
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.4 MB/s eta 0:00:00


In [ ]:
# test download a video
!yt-dlp --no-cache-dir --quiet --batch-file to_download.txt --paths downloaded/ -f "mp4"

In [ ]:
import os, re
from collections import defaultdict
from moviepy.editor import VideoFileClip
import moviepy.editor as mp
import matplotlib.pyplot as plt
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps
import torch
import torchaudio
import torchaudio.transforms as T
from IPython.display import Audio
import numpy as np
from PIL import Image
from paddleocr import PaddleOCR
import Levenshtein

from ppocr.utils.logging import get_logger
import logging
logger = get_logger()
logger.setLevel(logging.ERROR)

# use voice activity detector
model = load_silero_vad()

# Initialize PaddleOCR model
ocr = PaddleOCR(use_angle_cls=False, det = False, lang='ch')  # Adjust the language if necessary
# change to better model "rec_model_dir="
# Chinese Traditional	(chinese_cht)
# Chinese & English	(ch) -- got issue using
# English	(en)

In [ ]:
# create a class for extracting audio segments
class AudioExtractor:
    """
    Extract audio from a video file.
    Outputs the original audio as a .wav file.

    Audio segments can be extracted based on speech timestamps.
    Can play original audio segments by providing the start and end timestamps from resampled.
    """
    def __init__(self, video_fn, resample_rate=16000):
        self.audio_fn = video_fn.split(".mp4")[0]+".wav"
        self.resample_rate = resample_rate

        # convert video to audio using moviepy
        clip = VideoFileClip(video_fn)
        clip.audio.write_audiofile(self.audio_fn)

        # resample audio (required for extracting segments)
        self.waveform, self.sample_rate = torchaudio.load(self.audio_fn)
        self.waveform_mono = torch.mean(self.waveform, dim=0).unsqueeze(0)
        if self.sample_rate != self.resample_rate:
            resampler = T.Resample(self.sample_rate, self.resample_rate, dtype=self.waveform_mono.dtype)
            self.waveform_resampled = resampler(self.waveform_mono)
        else:
            self.waveform_resampled = self.waveform_mono

    def get_speech_timestamps(self, model):
        """
        Outputs a list of speech timestamps ecah in the form of dictionary {start: ..., end: ...}.
        """
        raw = get_speech_timestamps(self.waveform_resampled, model)
        # return only segments longer than 2 seconds
        # (some audio segments very short, like 0 to 1s)
        return [x for x in raw if (x['end'] - x['start'])/self.resample_rate > 2]

    def play_audio_segment(self, start, end):
        """
        Play in original audio sampling rate.
        More for self-checking.
        """
        # Step 1: Convert new timestamps to sample indices in the original audio
        original_start_index = int(start * self.sample_rate/self.resample_rate)
        original_end_index = int(end * self.sample_rate/self.resample_rate)

        # Step 2: Extract the corresponding segment from the original audio
        original_audio_segment = self.waveform[:,original_start_index:original_end_index]

        # Step 3: Play the extracted segment
        return Audio(original_audio_segment.numpy(), rate=self.sample_rate)

    def clean_audio(self):
        """
        Do we need to clean the audio like standardise/remove background noises?
        """
        return

In [ ]:
# file interested in
PATH = "/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk].mp4"

In [ ]:
# get audio segments
segmenting = AudioExtractor(PATH)
timestamps = segmenting.get_speech_timestamps(model)
timestamps

MoviePy - Writing audio in /content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk].wav


MoviePy - Done.


[{'start': 350240, 'end': 399328},
 {'start': 520224, 'end': 553440},
 {'start': 1256992, 'end': 1333216},
 {'start': 1572384, 'end': 1643488},
 {'start': 2104864, 'end': 2139616},
 {'start': 2385440, 'end': 2420704},
 {'start': 3897888, 'end': 3931104},
 {'start': 4499488, 'end': 4538336},
 {'start': 5337632, 'end': 5381600},
 {'start': 5488672, 'end': 5523936},
 {'start': 5585440, 'end': 5666784},
 {'start': 5852192, 'end': 5887456},
 {'start': 5903392, 'end': 5950432}]

In [ ]:
# play an audio segment
i = 3 ### testing: 0, 1
segmenting.play_audio_segment(timestamps[i]['start'], timestamps[i]['end'])

In [ ]:
# create class for extracting hard-coded subtitle
def play_video_segment(video_segment):
    """
    Play in original video sampling rate.
    More for self-checking.
    """
    return video_segment.ipython_display()

def apply_ocr(image_path, ocr):
    """
    Apply OCR to an image.
    """
    result = ocr.ocr(image_path, cls=False, det=False, rec=True)
    extracted_text = ""

    for line in result:
        for word_info in line:
            word = word_info[0]  # Extract the text
            extracted_text += word + " "

    return extracted_text.strip()

def calculate_similarity_levenshtein(sub1, sub2):
    """
    Calculate similarity based on Levenshtein distance.
    A lower distance means the subtitles are more similar.
    """
    distance = Levenshtein.distance(sub1, sub2)
    max_len = max(len(sub1), len(sub2))
    # Normalize the distance to get a similarity score between 0 and 1.
    # 0 means identical, 1 means completely different.
    return distance / max_len

def is_chinese_char(char):
    # Function to detect if a character is Chinese
    return '\u4e00' <= char <= '\u9fff'

def filter_non_chinese(text):
    # Remove non-Chinese characters
    return ''.join([char for char in text if is_chinese_char(char)])

class SubtitleExtractor:
    """
    Extract hard-coded subtitles from a video file.
    """
    def __init__(self, video_fn, resample_rate=16000):
        self.video_fn = video_fn
        self.video = mp.VideoFileClip(video_fn)
        self.resample_rate = resample_rate

    def get_video_segment(self, start, end):
        """
        Extract a video segment based on start and end timestamps.
        """
        # Step 1: Extract the video segment corresponding to the time range x (start seconds) to y (end seconds)
        video_segment = self.video.subclip(start/self.resample_rate, end/self.resample_rate)
        return video_segment

    def get_frames_of_cropped_subtitle_region(self, video_segment, i, frame_interval = 0.25):
        """
        One segment only.
        Extract frames containing subtitle at a regular interval.
        For each frame, crop the subtitle region (subtitles usually appear (e.g., the bottom part of the video)).
        Can possibly improve accuracy by focusing only on the subtitle area.

        frame_interval = 1 # (e.g., 1 frame per second)
        frame_interval = 0.5 # (e.g., 2 frames per second)
        frame_interval = 0.25 # (e.g., 4 frames per second)
        """
        paths_to_frames = []
        for t in np.arange(0, int(video_segment.duration), frame_interval):
            frame = video_segment.get_frame(t)
            # cropping
            frame = Image.fromarray(frame)
            width, height = frame.size
            subtitle_region = (0, int(0.8 * height), width, int(0.92 * height))
            # subtitle_region = (0, int(0.8 * height), width, height)  # bottom 20% of the frame
            cropped_image = frame.crop(subtitle_region)
            frame_image_path = f"{self.video_fn.split('.mp4')[0]}_{i}th_segment_cropped_frame_at_{t}_seconds.png"
            cropped_image.save(frame_image_path)
            paths_to_frames.append(frame_image_path)
        return paths_to_frames

    def extract_subtitles(self, cropped_image_paths, ocr):
        """
        One segement only.
        Extract subtitles from cropped frames.
        """
        subtitles = {}
        for p in cropped_image_paths:
            subtitle_text = apply_ocr(p, ocr)
            if subtitle_text:
                subtitles[p] = subtitle_text
                # print(f"Subtitle at {p.split('frame_at_')[-1]} seconds: {subtitle_text}")
        return subtitles

    def group_similar_subtitles(self, subtitles, similarity_threshold=0.4):
        """
        Group similar subtitles using Levenshtein distance.
        Subtitles with normalized distance below the threshold are grouped together.
        """
        grouped_subtitles = defaultdict(list)

        for path, subtitle in subtitles.items():
            # Clean up subtitle (e.g., remove non-Chinese if needed)
            filtered_subtitle = filter_non_chinese(subtitle)

            found_similar = False
            for key in grouped_subtitles:
                # Calculate Levenshtein distance similarity
                similarity = calculate_similarity_levenshtein(filtered_subtitle, key)
                if similarity < similarity_threshold:  # Group if the distance is small
                    grouped_subtitles[key].append(filtered_subtitle)
                    found_similar = True
                    break

            if not found_similar:
                grouped_subtitles[filtered_subtitle].append(filtered_subtitle)

        return grouped_subtitles

    def choose_best_subtitle(self, grouped_subtitles):
        """
        Currently implemented choose longest for same group. BUT OBVIOUSLY NOT BEST SOLUTION...
        Groups similar subtitles and concatenates different sequences.
        Uses Levenshtein distance to group similar ones.
        - Can consider:
            - Could use chinese LLM that says which text makes more sense (e.g., lower perplexity score)
            - other ways???
            - OCR for English in general more accurate than Chinese... but most videos only have chinese subtitles...
        """
        # grouped_subtitles = group_similar_subtitles(subtitles)
        final_subtitle = ""

        for key, group in grouped_subtitles.items():
            # For each group, choose the longest subtitle as the best one
            longest_subtitle = max(group, key=len)
            final_subtitle += longest_subtitle + " "

        return final_subtitle.strip()
        # best = ""
        # for t, subtitle in subtitles.items():
        #     if len(subtitle) > len(best):
        #         best = subtitle
        # return best

# # Generate frames as images
# for t in np.arange(0, int(video_segment.duration), frame_interval):
#     frame = video_segment.get_frame(t)
#     frame_image_path = f"{PATH.split('.mp4')[0]}_{i}th_segment_frame_at_{t}_seconds.png"
#     # Save frame as an image
#     mp.ImageClip(frame).save_frame(frame_image_path)

# def crop_subtitle_region(image_path):
#     image = Image.open(image_path)
#     width, height = image.size
#     subtitle_region = (0, int(0.8 * height), width, height)  # bottom 20% of the frame
#     cropped_image = image.crop(subtitle_region)
#     cropped_image_path = image_path.replace(".png", "_cropped.png")
#     cropped_image.save(cropped_image_path)
#     return cropped_image_path

# cropped_image_paths = [crop_subtitle_region(f"{PATH.split('.mp4')[0]}_{i}th_segment_frame_at_{t}_seconds.png") for t in np.arange(0, int(video_segment.duration), frame_interval)]
# len(cropped_image_paths)

In [ ]:
# Play corresponding video segment
extractsubtitle = SubtitleExtractor(PATH)
video_segment = extractsubtitle.get_video_segment(timestamps[i]['start'], timestamps[i]['end'])
play_video_segment(video_segment)

Moviepy - Building video __temp__.mp4.
MoviePy - Writing audio in __temp__TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video __temp__.mp4



Moviepy - Done !
Moviepy - video ready __temp__.mp4


In [ ]:
# extract cropped frames for that video segment
paths_to_frames = extractsubtitle.get_frames_of_cropped_subtitle_region(video_segment, i)
paths_to_frames

['/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk]_3th_segment_cropped_frame_at_0.0_seconds.png',
 '/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk]_3th_segment_cropped_frame_at_0.25_seconds.png',
 '/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk]_3th_segment_cropped_frame_at_0.5_seconds.png',
 '/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk]_3th_segment_cropped_frame_at_0.75_seconds.png',
 '/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk]_3th_segment_cropped_frame_at_1.0_seconds.png',
 '/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk]_3th_segment_cropped_frame_at_1.25_seconds.png',
 '/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk]_3th_segment_cropped_frame_at_1.5_seconds.png',
 '/content/downloaded/Teochew Comedy 9 - Hubby not Home (潮州搞笑-老公不在家） [bRovJsvW-lk]_3th_

In [ ]:
len(paths_to_frames)

16

In [ ]:
# Apply OCR
subtitles = extractsubtitle.extract_subtitles(paths_to_frames, ocr)
subtitles.values()

dict_values(['喉我得打电话跟她说', '喉我得打电话跟她说', '喉我得打电话跟她说', '喉我得打电话跟她说', '喉我得打电话跟她说', '喉我得打电话跟她说', '让她别喊了', '让她别喊了', '让她别喊了', '让她别喊了', '让她别喊了', '别等下让别人误会有什么', '别等下让别人误会有什么', '别等下让别人误会有什么', '别等下让别人误会有什么', '别等下让别人误会有什么'])

In [ ]:
len(subtitles) # for some frames, OCR fails to detect...

16

In [ ]:
grouped_subtitles = extractsubtitle.group_similar_subtitles(subtitles)
grouped_subtitles

defaultdict(list,
            {'喉我得打电话跟她说': ['喉我得打电话跟她说',
              '喉我得打电话跟她说',
              '喉我得打电话跟她说',
              '喉我得打电话跟她说',
              '喉我得打电话跟她说',
              '喉我得打电话跟她说'],
             '让她别喊了': ['让她别喊了', '让她别喊了', '让她别喊了', '让她别喊了', '让她别喊了'],
             '别等下让别人误会有什么': ['别等下让别人误会有什么',
              '别等下让别人误会有什么',
              '别等下让别人误会有什么',
              '别等下让别人误会有什么',
              '别等下让别人误会有什么']})

In [ ]:
# best subtitle (choose longest in same group to concatenate)
best_subtitle = extractsubtitle.choose_best_subtitle(grouped_subtitles)
best_subtitle

'喉我得打电话跟她说 让她别喊了 别等下让别人误会有什么'